In [ ]:
pip install -r requirements.txt

In [ ]:
import Prompts as prompts


In [ ]:
import importlib
importlib.reload(prompts)

In [2]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [3]:
from Prompts.prompts import Config

In [4]:
# Load your CSV
df_new = pd.read_csv("/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Amazon_Human_VS_ComputerFake.csv")
reviews = df_new["text"].tolist()

In [ ]:
prompt_template=Config.zero_shot_prompt_template

In [5]:
prompt_template=Config.zero_shot_prompt_template2

In [6]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [11]:
print(prompt_template)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements
    Review: "{text}"
    Classify this review as either "real" or "fake" (respond with only one word):
    


In [ ]:
apptainer --version

In [ ]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11400"

In [ ]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


In [9]:
df_new.head(10)


,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1
5,fake,Home_and_Kitchen,amazon,Bought this on a whim and it was the best purc...,1
6,fake,Home_and_Kitchen,amazon,"I admit I was leary of the suction, but I did ...",1
7,fake,Home_and_Kitchen,amazon,Purchased one as a gift and it was a very nice...,1
8,fake,Home_and_Kitchen,amazon,I wanted something to hold a bunch of pieces o...,1
9,fake,Home_and_Kitchen,amazon,I'm so happy I found this set and am very plea...,1


In [13]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [ ]:
y_true = df_new["Binary_label"].tolist()

In [ ]:
for label in results:
    print(label)

In [49]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS
Accuracy: 0.5675

F1 Score: 0.4933780410129511

Confusion Matrix:
[[ 74 326]
 [ 20 380]]


In [10]:
# RESULTS WITH PROMPT 2 FOR ZERO SHOT : llama 3.2 :3B
accuracy, f1, conf_matrix = evaluate_model(df_new, results)
print(Config.model)
print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS WITH PROMPT 2")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.2:3b
ZERO-SHOT PROMPTING RESULTS WITH PROMPT 2
Accuracy: 0.635

F1 Score: 0.6348539415766307

Confusion Matrix:
[[246 154]
 [138 262]]


In [9]:
# Results with LLAMA 3.1 8B
print(Config.model)
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS With LLAMA 3.1:8B")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.1:8b
ZERO-SHOT PROMPTING RESULTS With LLAMA 3.1:8B
Accuracy: 0.68375

F1 Score: 0.6700525405465325

Confusion Matrix:
[[192 208]
 [ 45 355]]


In [14]:
# Results with LLAMA 3.1 8B prompt 2
print(Config.model)
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS With LLAMA 3.1:8B  PROMPT 2")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

llama3.1:8b
ZERO-SHOT PROMPTING RESULTS With LLAMA 3.1:8B  PROMPT 2
Accuracy: 0.7225

F1 Score: 0.7201051496870096

Confusion Matrix:
[[252 148]
 [ 74 326]]


## One-Shot Prompting
Using one example to guide the model's classification

In [5]:
llm = OllamaLLM(model=Config.model)

In [ ]:
prompt_template_oneshot=Config.one_shot_prompt_template

In [6]:
prompt_template_oneshot=Config.one_shot_prompt_template2

In [7]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = prompt_template_oneshot
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [8]:
print(prompt_template_oneshot)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    Task: Classify the following review of a  Amazon product as either "real" or "fake". Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are sup

In [ ]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


In [ ]:
df_new.head(10)

In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [12]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS
Accuracy: 0.6175

F1 Score: 0.616

Confusion Matrix:
[[272 128]
 [178 222]]


In [16]:
#One shot results with LLAMA 3.1 8B
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)
print(Config.model)
print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.1 8B")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

llama3.1:8b
ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.1 8B
Accuracy: 0.68875

F1 Score: 0.6828626422700157

Confusion Matrix:
[[221 179]
 [ 70 330]]


In [13]:
#One shot results with LLAMA 3.2 3B Prompt 2
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)
print(Config.model)
print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.2 3B WITH PROMPT 2")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

llama3.2:3b
ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.2 3B WITH PROMPT 2
Accuracy: 0.5425

F1 Score: 0.5028220958901318

Confusion Matrix:
[[104 296]
 [ 70 330]]


In [11]:
#One shot results with LLAMA 3.1 8B Prompt 2
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)
print(Config.model)
print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.1 8B WITH PROMPT 2")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

llama3.1:8b
ONE-SHOT PROMPTING RESULTS WITH LLAMA 3.1 8B WITH PROMPT 2
Accuracy: 0.655

F1 Score: 0.6518580186180277

Confusion Matrix:
[[224 176]
 [100 300]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [5]:
llm = OllamaLLM(model=Config.model)

In [6]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template2
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [7]:
print(Config.few_shot_prompt_template2)


    You are an expert at detecting real and fake reviews.


    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Example 1:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are super lightweight and lightweight. I have used them for a couple of months and they are still working gr

In [ ]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        cleaned_label = "fake"
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


In [12]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS
Accuracy: 0.63375

F1 Score: 0.6040801366126218

Confusion Matrix:
[[363  37]
 [256 144]]


In [21]:
# Results for few shot llama 3.1 8b
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)
print(Config.model)
print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS FOR LLAMA 3.1:8B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

llama3.1:8b
FEW-SHOT PROMPTING RESULTS FOR LLAMA 3.1:8B
Accuracy: 0.705

F1 Score: 0.7017113678303294

Confusion Matrix:
[[324  76]
 [160 240]]


In [10]:
# Results for few shot WITH ADDITIONAL EXAMPLES llama  3.1 8b
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)
print(Config.model)
print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS WITH ADDITIONAL EXAMPLES FOR LLAMA 3.1:8B")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

llama3.1:8b
FEW-SHOT PROMPTING RESULTS WITH ADDITIONAL EXAMPLES FOR LLAMA 3.1:8B
Accuracy: 0.70875

F1 Score: 0.7023842787369213

Confusion Matrix:
[[342  58]
 [175 225]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")

## Results With context for categories


In [ ]:
pip install -r requirements.txt

In [2]:
from Prompts.prompt_category_context import Config

In [3]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.

    Product Category:
    {category}

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [6]:
# Load your CSV
df_new = pd.read_csv("/home/sohy47ma/ReviewProject_new/fake_reviews_prediction/Datasets/Amazon_Human_VS_ComputerFake.csv")
reviews = df_new["text"].tolist()
categories = df_new["Category"].tolist()

### Zero shot


In [7]:
prompt = PromptTemplate(
    input_variables=["text","category"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [ ]:
# Classify each review
results = []

for r, c in zip(reviews, categories):
    result = chain.invoke({"text": r, "category": c})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


In [9]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [10]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.5425

F1 Score: 0.4676363636363636

Confusion Matrix:
[[ 67 333]
 [ 33 367]]


## One Shot

In [11]:
llm = OllamaLLM(model=Config.model)

In [12]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfe

In [13]:
one_shot_prompt = PromptTemplate(
    input_variables=["text","category"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [ ]:
# Classify each review
one_shot_results = []

for r, c in zip(reviews, categories):
    result = chain.invoke({"text": r, "category": c})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


In [15]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.54125

F1 Score: 0.46676256222768286

Confusion Matrix:
[[ 67 333]
 [ 34 366]]


## Few Shot

In [16]:
llm = OllamaLLM(model=Config.model)

In [17]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfe

In [18]:
few_shot_prompt = PromptTemplate(
    input_variables=["text","category"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [ ]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r,"category": c})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


In [20]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.575

F1 Score: 0.5086385675369063

Confusion Matrix:
[[377  23]
 [317  83]]
